[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/02_%E7%9B%B8%E9%96%A2%E3%81%A8%E5%9B%9E%E5%B8%B0.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計3級-02. 相関と回帰直線

**できるようになること**
- 散布図と相関係数で「関係の強さ」を読める
- 相関≠因果（擬相関）を説明できる
- 回帰直線でxからyを予測できる

**前提**：`01_Python基礎`・統計3級01　/　**所要**：約30分　/　**レベル**：統計検定3級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む
df.head()                        # 先頭5行を確認

In [ ]:
# ここに書いて試してみよう

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

## 1. 散布図

2つの量的データの関係を点で表します。ここでは「数学」と「英語」の関係を見ます。

In [ ]:
plt.figure(figsize=(6, 5))                  # グラフの大きさ
plt.scatter(df['数学'], df['英語'], alpha=0.6)  # 散布図（数学×英語）
plt.xlabel('数学')                          # x軸ラベル
plt.ylabel('英語')                          # y軸ラベル
plt.title('数学と英語の関係')               # タイトル
plt.show()

In [ ]:
# ここに書いて試してみよう

## 2. 相関係数 r

関係の強さを −1〜+1 の数で表します。

| r の値 | 関係 |
|---|---|
| +0.7〜+1.0 | 強い正の相関 |
| +0.4〜+0.7 | やや正の相関 |
| −0.2〜+0.2 | ほぼ無相関 |
| −1.0〜−0.7 | 強い負の相関 |

In [ ]:
r = df['数学'].corr(df['英語'])             # 数学と英語の相関係数
print('相関係数 r =', round(r, 3))

# 全教科の相関行列（各ペアの相関をまとめて表示）
df[['数学', '英語', '国語', '勉強時間']].corr().round(2)

In [ ]:
# ここに書いて試してみよう

> **相関 ≠ 因果**。「アイスの売上」と「水難事故」は相関するが、
原因は両方とも「気温（夏）」。見かけの相関に注意！

## 3. 回帰直線（最小二乗法）

点の真ん中を通る直線 `y = a·x + b` を引き、xからyを予測します。

In [ ]:
x = df['数学']                             # 説明変数（x）
y = df['英語']                             # 目的変数（y）
a, b = np.polyfit(x, y, 1)   # 最小二乗法で 傾きa・切片b を求める
print(f'回帰直線: y = {a:.2f} x + {b:.2f}')

plt.figure(figsize=(6, 5))                 # グラフの大きさ
plt.scatter(x, y, alpha=0.5)               # 散布図
xs = np.linspace(x.min(), x.max(), 100)    # 直線を描くためのx
plt.plot(xs, a * xs + b, color='red', label=f'y={a:.2f}x+{b:.2f}')  # 回帰直線
plt.xlabel('数学'); plt.ylabel('英語'); plt.legend()
plt.show()

In [ ]:
# ここに書いて試してみよう

### 予測してみる
数学が85点の人の英語を予測すると？

In [ ]:
pred = a * 85 + b                          # 回帰式に数学85を入れて英語を予測
print(f'数学85点 → 英語の予測 {pred:.1f}点')

In [ ]:
# ここに書いて試してみよう

> **よくある間違い**：相関があっても**因果とは限らない**（裏に共通の原因＝擬相関）。また相関係数が測るのは**直線関係の強さ**だけで、曲線の関係は0に近く出ることがあります。

## 4. 相関係数は「単位」を変えても変わらない

相関係数は、各変数を **a倍して b を足す**ような一次変換（単位の変更）をしても変わりません。標準化（zスコア化）も一次変換なので、相関は元のままです。

In [ ]:
m, e = df['数学'], df['英語']
print('元の r            :', round(m.corr(e), 3))
print('数学を×10+5, 英語を÷2:', round((m*10+5).corr(e/2), 3))   # 一次変換 → 不変
zm, ze = (m-m.mean())/m.std(), (e-e.mean())/e.std()
print('両方を標準化(zスコア):', round(zm.corr(ze), 3))            # これも不変

In [ ]:
# ここに書いて試してみよう

> ただし「**1人あたり**」「**千人あたり**」のように別の量で割る変換は一次変換ではありません（非線形）。この手の加工では相関係数が変わることがあるので注意します。

## 5. 外れ値1つで相関はガラリと変わる

きれいな正の相関でも、極端な点が1つ混じるだけで相関係数は大きく動きます。

In [ ]:
xs = np.array([10,20,30,40,50,60,70])
ys = np.array([12,19,33,38,52,61,68])         # きれいな正の相関
xs2, ys2 = np.append(xs, 66), np.append(ys, 6)  # 右下に外れ値を1つ追加
print('外れ値なし r =', round(np.corrcoef(xs, ys)[0,1], 3))
print('外れ値1つ  r =', round(np.corrcoef(xs2, ys2)[0,1], 3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
a1.scatter(xs, ys); a1.set_title(f'なし r={np.corrcoef(xs,ys)[0,1]:.2f}')
a2.scatter(xs2, ys2); a2.scatter([66],[6], color='red', s=80, label='外れ値')
a2.set_title(f'1つ追加 r={np.corrcoef(xs2,ys2)[0,1]:.2f}'); a2.legend()
plt.show()

In [ ]:
# ここに書いて試してみよう

> 相関係数を見る前に**散布図を描く**のが鉄則です。数字だけ見ていると外れ値に気づけません。

## 6. シンプソンのパラドックス（集計の落とし穴）

**層（グループ）ごとに見ると正の相関なのに、全部まとめると負の相関に見える**——そんな逆転が実際に起こります。架空の「部署別：研修時間と成約率」で再現します。

In [ ]:
rng = np.random.default_rng(3)
# 部署ごとに『研修時間が多いほど成約率が高い』(正の相関)。
# ただし部署Aは研修少なめでも元々成約率が高く、部署Cは研修多めでも低い。
centers = {'部署A': (2, 55), '部署B': (5, 40), '部署C': (8, 25)}  # (研修時間の中心, 成約率の中心)
parts = []
for g, (cx, cy) in centers.items():
    x = rng.normal(cx, 0.8, 40)
    y = cy + 2.5 * (x - cx) + rng.normal(0, 2, 40)   # 群内の傾きは +2.5（正）
    parts.append(pd.DataFrame({'部署': g, '研修時間': x, '成約率': y}))
sim = pd.concat(parts, ignore_index=True)
print('全体をまとめた相関 :', round(sim['研修時間'].corr(sim['成約率']), 3))
for g in centers:
    d = sim[sim['部署'] == g]
    print(f'  {g} だけの相関   :', round(d['研修時間'].corr(d['成約率']), 3))

In [ ]:
# ここに書いて試してみよう

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
colors = {'部署A':'C0', '部署B':'C1', '部署C':'C2'}
for g in centers:
    d = sim[sim['部署'] == g]
    ax.scatter(d['研修時間'], d['成約率'], color=colors[g], alpha=.6, label=g)
    a, b = np.polyfit(d['研修時間'], d['成約率'], 1)
    xx = np.linspace(d['研修時間'].min(), d['研修時間'].max(), 20)
    ax.plot(xx, a*xx + b, color=colors[g])               # 群ごとの回帰線（正の傾き）
A, B = np.polyfit(sim['研修時間'], sim['成約率'], 1)
xx = np.linspace(sim['研修時間'].min(), sim['研修時間'].max(), 50)
ax.plot(xx, A*xx + B, 'k--', lw=2, label=f'全体 (傾き{A:.1f})')  # 全体の回帰線（負の傾き）
ax.set_xlabel('研修時間'); ax.set_ylabel('成約率'); ax.set_title('シンプソンのパラドックス')
ax.legend(); plt.show()

In [ ]:
# ここに書いて試してみよう

> **層を無視して全体だけ見ると、逆の結論になる**ことがあります（シンプソンのパラドックス）。部署ごとには「研修するほど成約率が上がる」のに、まとめると「研修するほど下がる」ように見える。
>
> データを集計する前に、**分けて見るべきグループ（層）がないか**を必ず確認しましょう。これは実務でいちばんやりがちな誤読です。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import numpy as np
x = [1,2,3,4,5]; y = [2,4,6,8,10]
# Q1: x と y の相関係数を ans に（完全な正の相関）
ans = None   # 例: np.corrcoef(x, y)[0,1]
_check('Q1 相関係数', ans, np.corrcoef(x,y)[0,1])

In [ ]:
import numpy as np
x = [1,2,3,4,5]; y = [2,4,6,8,10]
# Q2: y = a*x + b の傾き a を ans に
ans = None   # 例: np.polyfit(x, y, 1)[0]
_check('Q2 回帰の傾き', ans, np.polyfit(x,y,1)[0])

In [ ]:
import numpy as np
a = [2,4,6,8,10]; b = [1,3,2,5,4]
# Q3: a を「×100」しても相関係数は変わらない。変換後の相関を ans に
ans = None   # 例: np.corrcoef(np.array(a)*100, b)[0,1]
_check('Q3 線形変換後の相関', ans, np.corrcoef(np.array(a)*100, b)[0,1])

---
## 練習問題

**問1.** `勉強時間` と `数学` の散布図を描き、相関係数を求めよう。正の相関はある？

**問2.** `勉強時間`(x) から `数学`(y) を予測する回帰直線を求め、
「3時間勉強した人」の数学点数を予測しよう。

**問3.** `国語` と `数学` の相関係数を求めよう。関係は強い？弱い？

**問4.** 上で作った `sim` データで、(1) 全体の相関 (2) 部署ごとの相関 を計算し、符号が逆になることを確かめよう。なぜ逆転するのか、1行で説明してみよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[02_相関と回帰 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/02_%E7%9B%B8%E9%96%A2%E3%81%A8%E5%9B%9E%E5%B8%B0.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 散布図 | 2つの量を点で表す図 |
| 相関係数 r | 直線関係の強さ（−1〜1） |
| 擬相関 | 共通原因による見かけの相関 |
| 最小二乗法 | 点に最も近い直線を引く方法 |
| 回帰係数 | 回帰直線の傾き・切片 |

In [ ]:
# チートシート（相関・回帰）
df['数学'].corr(df['英語'])        # 相関係数
df[['数学','英語','国語']].corr()   # 相関行列
import numpy as np
a, b = np.polyfit(df['数学'], df['英語'], 1)   # 回帰直線 傾きa・切片b
plt.scatter(df['数学'], df['英語'])           # 散布図

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "03_統計_3級/02_相関と回帰"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")